# HW4 B: Automatic speech recognition (30 automatic + 3 report points)
Build a HuBERT + downsampling convolution + LSTM CTC recognizer.
Use the compact LibriSpeech subset supplied with the course. Training and evaluation
speakers are disjoint. Test transcripts are withheld from the student package.

**Grading:** CTC output lengths (2), CTC loss (3), greedy decoding (2), model (5), and test WER (18).
Test WER points: ≤50%: 18; ≤60%: 15; ≤70%: 12; ≤80%: 9; ≤90%: 6; ≤99%: 3; otherwise: 0.
These generous bands use a staff reference benchmark of about 29% test WER.
The 12 implementation points are independent of training progress.
These are corpus-level WER values; missing predictions count as deletions.
Use validation WER for model selection. Answer report question B1 in the shared PDF; the implementation and predictions are autograded.

**Written answers:** submit one searchable, text-selectable `report.pdf` using the
five section IDs **A1, A2, A3, B1, C1** in `report_template.txt`. Put conceptual
answers and requested figures in that PDF; notebook markdown answers are not
manually graded. Aim for 3–5 sentences per section, plus the requested figures or
token sequence. There is no minimum page count. No extra experiments are required
for B1/C1. Mark unfinished work `Not completed` and explain what is available.
The shared report is worth 15 points (A: 9, B: 3, C: 3); the autograder is worth 85.


## Setup
Open this notebook on Colab and select a GPU. Setup automatically downloads the
small support bundle from the public course Hugging Face repository; no account,
token, or TA Drive access is needed. The next cell downloads only this part's data.
Downloads use a fixed release and verify checksums. Setup installs Python
dependencies while preserving Colab's matched PyTorch/CUDA installation.
Run setup once in a fresh runtime. If pip asks for a restart, restart and rerun setup.

Implement each TODO before running the cells that use it. For a quick check of
**your implementation**, set `SMOKE_TEST = True`. Smoke results are not completed experiments.
Definitions used by the autograder must be top-level functions/classes with the
specified names and signatures. The grader provides the standard imports shown
in setup; put any additional imports inside your functions. Keep helper constants
inside functions or as class attributes: top-level assignments are not executed.
A syntax error skips that cell; other independently implemented questions still count.

### What you must complete

1. Implement `ctc_output_lengths`, `ctc_loss`, and `greedy_decode` using the shapes
   and blank/padding conventions below. For a quick decoder check,
   `[blank,B,B,blank,B,A,A]` must produce `BBA`; padding after the valid length
   must not change it. Loss must be a finite scalar with gradients.
2. Complete `ASR.__init__` and `ASR.encode`. The provided `forward`/`predict`
   methods call your helpers. Freeze the HuBERT convolutional frontend; by
   default the remaining HuBERT layers and the new CTC layers are trainable.
   `encode` returns raw `[batch,time,vocab]` logits and one valid length per item.
3. Run the supplied training loop, initially in smoke mode, then normal mode.
   Data loading, optimizer groups, validation WER, checkpoint saving/resumption,
   and test export are provided. You do not need to write another training loop.
   Start with the supplied settings and select checkpoints by validation WER.
4. Run **Export test predictions** using your best saved validation checkpoint.
   The full output is **95** `audio_id|prediction` lines in
   `RESULTS/asr_submission.txt`. Do not insert reference transcripts or manually
   edit recognition output. If evaluation stops, rerun it to resume.
5. Answer B1 in the shared PDF and run the final ZIP cell. The checkpoint stays
   on your Drive/local runtime and is never submitted. Missing test utterances
   count as deletions; code checks remain independent of training progress.

Training can predict only blanks initially. First verify valid sequence lengths,
finite loss, and changing validation loss before treating early 100% WER as a
bug. A smoke run checks execution; it is not a completed model-performance run.


In [ ]:
REQUIREMENTS = 'requirements.txt'
# Colab: select Runtime > Change runtime type > GPU before training.
import sys, subprocess
from pathlib import Path

# Course release settings are supplied by staff; no student login is needed.
COURSE_REPO_ID = 'Louis0324/eecs183-hw4'
COURSE_REVISION = 'hw4-2026-v2'
import os
os.environ.setdefault("HW4_HF_REPO", COURSE_REPO_ID)
os.environ.setdefault("HW4_HF_REVISION", COURSE_REVISION)

def ensure_hw4_support(repo_id=None, revision='hw4-2026-v2', packager_only=False):
    """Fetch the public support bundle, or use a local/uploaded offline copy."""
    import hashlib
    from io import BytesIO
    import json
    from pathlib import Path
    import re
    from urllib.parse import quote
    from urllib.request import urlopen
    import zipfile

    required = 'make_submission.py' if packager_only else 'course_download.py'
    if Path(required).is_file():
        return
    local_bundle = Path('hw4-student.zip')
    if local_bundle.is_file():
        content = local_bundle.read_bytes()
    elif repo_id:
        if not re.fullmatch(r'[\w.-]+/[\w.-]+', repo_id):
            raise ValueError('Expected a Hugging Face repository ID like COURSE_ORG/eecs183-hw4.')
        base = f'https://huggingface.co/datasets/{repo_id}/resolve/{quote(revision, safe="")}/'
        print('Downloading the small HW4 support bundle from Hugging Face; no login is required.')
        try:
            with urlopen(base + 'release_manifest.json', timeout=30) as response:
                manifest = json.loads(response.read(1024 * 1024))
            spec = manifest['files']['hw4-student.zip']
            if not 0 < spec['bytes'] < 30 * 1024 * 1024:
                raise ValueError('Invalid support bundle size in the release manifest.')
            with urlopen(base + 'hw4-student.zip', timeout=30) as response:
                content = response.read(spec['bytes'] + 1)
            if len(content) != spec['bytes'] or hashlib.sha256(content).hexdigest() != spec['sha256']:
                raise ValueError('Support bundle checksum mismatch; rerun to download a fresh copy.')
        except Exception as error:
            raise RuntimeError('Could not download the HW4 support release. Rerun this cell; '
                               'if needed, download hw4-student.zip from the course Hugging Face repo '
                               'and upload it through the Files sidebar before retrying.') from error
    else:
        # Allows local staff testing and offline/manual use before publication.
        try:
            from google.colab import files
        except ImportError as error:
            raise FileNotFoundError('Put hw4-student.zip in this folder or configure the course Hugging Face repository.') from error
        print('Upload hw4-student.zip to load the supplied helpers.')
        uploaded = files.upload()
        archives = [value for name, value in uploaded.items() if name.lower().endswith('.zip')]
        if len(archives) != 1:
            raise ValueError('Upload exactly one hw4-student.zip support bundle, then rerun.')
        content = archives[0]

    with zipfile.ZipFile(BytesIO(content)) as archive:
        if archive.testzip() is not None:
            raise ValueError('Corrupt support ZIP; download a fresh copy.')
        for item in archive.infolist():
            path = Path(item.filename)
            if (path.is_absolute() or '..' in path.parts or
                    ((item.external_attr >> 16) & 0o170000) == 0o120000 or
                    not path.resolve().is_relative_to(Path.cwd().resolve())):
                raise ValueError('Unsafe support archive path.')
            # Browser/local notebook edits must never be replaced by starter copies.
            if item.is_dir() or path.suffix == '.ipynb' or (packager_only and item.filename != required):
                continue
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(archive.read(item))
    if not Path(required).is_file():
        raise ValueError(f'This ZIP is missing {required}; use the current hw4-student.zip support bundle.')

ensure_hw4_support(COURSE_REPO_ID, COURSE_REVISION)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', REQUIREMENTS])
import torch
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SMOKE_TEST = False  # True: only a few steps, for debugging your implementation.
DATA_ROOT = Path(os.environ.get('HW4_DATA_ROOT', 'data'))
RESULTS = Path('results'); RESULTS.mkdir(exist_ok=True)
# Checkpoints/results disappear when Colab deletes a runtime. Copy them to Drive
# or download them before disconnecting. Submission ZIPs contain no checkpoints.

In [ ]:
# Download only Part B data; verified cached copies are reused.
from course_download import ensure_course_data
DATA_ARCHIVE = None  # Optional offline override: Path("hw4-data-b.zip")
ensure_course_data('b', data_root=DATA_ROOT, archive=DATA_ARCHIVE)


In [ ]:
import json, math, copy
import numpy as np
import torch
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence, unpad_sequence
from sklearn.metrics import accuracy_score, r2_score
import matplotlib.pyplot as plt
from hw4_runtime import load_phoneme_data, load_mfcc_data, load_audio
np.random.seed(183); torch.manual_seed(183)
device = DEVICE

from hw4_runtime import CharacterTokenizer, SpeechTextDataset, train_asr_epoch, evaluate_asr, save_checkpoint
DATA_DIR = DATA_ROOT / 'asr'
dataset = SpeechTextDataset(DATA_DIR, 'train')
print('Training utterances:', len(dataset)); print(dataset[0]['text'])

## CTC building blocks
CTC has no frame-level text labels. The loss sums over valid alignments between
frame predictions and the transcript. Padding is not part of the audio: calculate
each sequence's encoded length separately.

Implement these helpers and use them in your model. HuBERT-base has convolution
kernels `[10,3,3,3,3,2,2]` and strides `[5,2,2,2,2,2,2]`, without padding.
The added convolution uses kernel 2, stride 2. At each stage the length is
`floor((L-kernel)/stride)+1`. Apply log-softmax before CTC and transpose logits
from `[batch,time,vocab]` to `[time,batch,vocab]`. Blank ID is 0; padding ID is 1.

In [ ]:
def ctc_output_lengths(waveform_lengths):
    """Compute valid frame counts after all seven HuBERT convolutions and the kernel-2/stride-2 downsampler. Input: int tensor [B] of waveform lengths; output: int64 [B], clamped at zero."""
    raise NotImplementedError('TODO: ctc_output_lengths')

In [ ]:
def ctc_loss(logits, targets, input_lengths, target_lengths, blank_id=0):
    """Input unnormalized logits [B,T,V], padded target IDs [B,U], input_lengths [B], target_lengths [B]. Return scalar mean torch CTC loss with blank_id and zero_infinity=True. Apply log_softmax, transpose, preserve gradients. Padding targets are excluded by target_lengths."""
    raise NotImplementedError('TODO: ctc_loss')

In [ ]:
def greedy_decode(logits, lengths, tokenizer):
    """Argmax logits [B,T,V], truncate each row to lengths[b], collapse adjacent repeated IDs, THEN remove blank/pad through tokenizer.decode. Return a list of B strings. Blanks separate repeated letters: A blank A -> AA."""
    raise NotImplementedError('TODO: greedy_decode')

## Implement the model
Use `facebook/hubert-base-ls960`, freeze its convolutional feature extractor, and
keep the LSTM modest (for example, 128 hidden units and 1–2 bidirectional layers).
You may use a different HuBERT layer, but report the choice. Greedy decoding is
provided as an exercise to avoid a native Flashlight dependency on Colab.

In [ ]:
from hw4_runtime import load_hubert
class ASR(nn.Module):
    def __init__(self, upstream=None, hidden_size=128, num_layers=2):
        super().__init__()
        self.token_processor = CharacterTokenizer()
        # Pass an upstream explicitly for small offline tests.
        self.upstream = upstream if upstream is not None else load_hubert()
        self.upstream.feature_extractor._freeze_parameters()
        # TODO: create downsample Conv1d(768, 128, kernel_size=2, stride=2),
        # a batch_first bidirectional LSTM, and a final Linear to vocab size.
        raise NotImplementedError('TODO: ASR layers')

    def encode(self, input_values, attention_mask):
        """Return (raw logits [B,T,V], valid lengths [B]).
        Run HuBERT, transpose for Conv1d and back, run the LSTM and output layer.
        Use packed sequences for a bidirectional LSTM so padding cannot affect
        backward states. Calculate lengths with ctc_output_lengths.
        """
        raise NotImplementedError('TODO: ASR.encode')

    def forward(self, input_values, text, attention_mask=None):
        if attention_mask is None:
            attention_mask = torch.ones_like(input_values, dtype=torch.long)
        logits, lengths = self.encode(input_values, attention_mask)
        targets, target_lengths = self.token_processor.encode_torch_batch(text)
        return ctc_loss(logits, targets.to(input_values.device), lengths, target_lengths)

    @torch.no_grad()
    def predict(self, input_values, attention_mask=None):
        if attention_mask is None:
            attention_mask = torch.ones_like(input_values, dtype=torch.long)
        logits, lengths = self.encode(input_values, attention_mask)
        return greedy_decode(logits, lengths, self.token_processor), logits

## Train and choose a checkpoint
Start with batch size 2 on a T4. If memory is tight, reduce the batch size or freeze
HuBERT initially. Loss may improve before greedy WER improves. Save checkpoints
outside ephemeral Colab storage if you need to resume later. The code saves both
the latest state and the best validation state; test predictions use the latter.
No specific training time or final WER is guaranteed for the compact dataset.
The provided optimizer uses a larger learning rate for the randomly initialized
CTC layers than for pretrained HuBERT. CTC often predicts blanks early in training;
do not stop solely because WER initially stays at 100%. Save and resume checkpoints.


In [ ]:
if device.type != 'cuda':
    raise RuntimeError('Select a GPU runtime before training.')
batch_size, epochs = 2, (1 if SMOKE_TEST else 20)
train_dataset = SpeechTextDataset(DATA_DIR, 'train')
val_dataset = SpeechTextDataset(DATA_DIR, 'dev')
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, collate_fn=SpeechTextDataset.collate)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, collate_fn=SpeechTextDataset.collate)
model = ASR().to(device)
from hw4_runtime import make_asr_optimizer
optimizer = make_asr_optimizer(model, upstream_lr=1e-5, head_lr=1e-3)
best_wer, start_epoch = float('inf'), 0
RESUME = False
if RESUME and (RESULTS / 'asr_latest.pt').exists():
    state = torch.load(RESULTS / 'asr_latest.pt', map_location='cpu', weights_only=True)
    model.load_state_dict(state['model_state_dict']); optimizer.load_state_dict(state['optimizer_state_dict'])
    start_epoch, best_wer = state['epoch'] + 1, state['best_wer']
for e in range(start_epoch, epochs):
    loss = train_asr_epoch(model, train_dataloader, optimizer, device, max_steps=3 if SMOKE_TEST else None)
    val_wer = evaluate_asr(model, val_dataloader, device, max_steps=2 if SMOKE_TEST else None)
    print(f'Epoch {e+1}: loss={loss:.4f}, validation WER={val_wer:.2%}')
    if val_wer < best_wer:
        best_wer = val_wer
        save_checkpoint(RESULTS / 'asr_best.pt', model, epoch=e, best_wer=best_wer)
    save_checkpoint(RESULTS / 'asr_latest.pt', model, optimizer, epoch=e, best_wer=best_wer)

## Export test predictions
The test dataset intentionally has no transcripts. Save one `audio_id|prediction`
line per utterance. The exporter saves progress after each utterance; rerun to resume.
When the model weights change, the exporter archives old predictions and starts
a fresh file. Matching checkpoints resume their completed predictions. Set `RESTART_PREDICTIONS=True`
if you changed decoding/generation code or settings and want to reevaluate from
the beginning; set it back to `False` to resume an interrupted evaluation.


In [ ]:
from hw4_runtime import load_transcription
state = torch.load(RESULTS / 'asr_best.pt', map_location='cpu', weights_only=True)
model.load_state_dict(state['model_state_dict']); model.eval()
test_dataset = SpeechTextDataset(DATA_DIR, 'test')
output = RESULTS / 'asr_submission.txt'
from hw4_runtime import model_fingerprint
RESTART_PREDICTIONS = False  # True after changing decoding code/settings.
signature = model_fingerprint(model)
signature_file = RESULTS / 'asr_prediction_model.txt'
if output.exists() and (RESTART_PREDICTIONS or not signature_file.exists() or signature_file.read_text() != signature):
    output.replace(RESULTS / 'asr_submission.previous.txt')
    print('Starting new predictions; the previous file was archived.')
signature_file.write_text(signature)
predictions = load_transcription(output) if output.exists() else {}
order = (DATA_DIR / 'test_submission_order.txt').read_text().splitlines()
with torch.no_grad():
    for i in range(min(3, len(test_dataset)) if SMOKE_TEST else len(test_dataset)):
        sample = test_dataset[i]
        if predictions.get(sample['id']):
            continue
        pred, _ = model.predict(sample['wav'][None].to(device))
        predictions[sample['id']] = pred[0].replace('\n', ' ')
        temporary = output.with_suffix('.tmp')
        temporary.write_text(''.join(f'{key}|{predictions.get(key, "")}\n' for key in order))
        temporary.replace(output)
print('Saved', output)

## Report B1: CTC decoding and evaluation (3 manual points)
Put this answer in the **B1** section of `report.pdf`. Use about 3–5 sentences plus
the short decoded sequence. This question needs no additional training.

1. For the frame path `[A, A, blank, A, B, B, blank]`, give the greedy CTC transcript
   and explain the order of collapsing repeats and removing blanks. (1 point)
2. Explain why padded frames must be excluded using each utterance's valid encoded
   length, for both CTC loss and decoding. (1 point)
3. Explain why checkpoint selection uses validation WER instead of test WER. (1 point)


## Prepare your submission
**Default: submit one combined ZIP containing one shared `report.pdf`.**
The PDF covers **A1, A2, A3, B1, C1** (and BONUS if attempted). Use
`report_template.txt`, keep all five headings, and write `Not completed` only for
answers you did not finish. You may answer B1/C1 without training those models.
Export one PDF with selectable text from Google Docs, Word, or LaTeX. If you wrote
separate reports, combine them into this document before uploading; the helper
**does not merge PDFs**. Notebook PDFs are not required.

**Typical A → B → C workflow**

1. **After A:** run its final cell. Download `submission.zip` and rename it
   `hw4-a.zip` on your computer. At the report prompt, upload your current combined
   PDF or cancel if it is not ready.
2. **After B:** leave `include_previous=True` (B's default). At the **earlier ZIP**
   prompt, upload `hw4-a.zip`. At the separate **report** prompt, upload an updated
   combined PDF, or cancel to keep the report already in the ZIP. The cell adds
   your current B notebook and available B predictions. Save the download as
   `hw4-ab.zip`.
3. **After C:** leave `include_previous=True` (C's default). Upload `hw4-ab.zip`
   at the earlier-ZIP prompt, then your final combined `report.pdf` at the report
   prompt. The cell adds C and downloads the final `submission.zip`. Upload
   **that one ZIP** to the HW4 Programming Assignment on Gradescope.

**Stopping early or working in another order**

| Work you want graded | What to do |
|---|---|
| A + B only | Finish step 2 with your final report, then submit `hw4-ab.zip`. You do **not** need to open or run C. |
| A only, or part of A | Submit A's ZIP with the report answers you completed. |
| Only B or C | Cancel the earlier-ZIP prompt; the cell still packages the current notebook and available results. |
| Separate A and B ZIPs, finishing C | Select both A and B ZIPs together at C's earlier-ZIP prompt, then upload the shared report at the next prompt. |
| Another order, or updating earlier work | Set `include_previous=True` in whichever notebook you package last and import your latest combined ZIP. Current notebook edits and available local results replace their earlier copies. |

Use the **latest** ZIP for each part; do not select both an old and a new version
of the same part. Keep `include_report=True`: cancelling its upload preserves an
existing report; setting it to `False` deliberately leaves the report out.
Missing code, predictions, or a PDF do not block grading of other completed work.
Missing C does not reduce A/B credit. Save each ZIP before disconnecting Colab.

**Before uploading:** read the printed notebook summary, file list, and warnings.
Check that all notebooks you want graded and their available prediction files
are included. Open `report.pdf` inside the ZIP to confirm it is your latest report.
Warnings for intentionally unfinished questions are expected; fix warnings about
work you intended to include. Every Gradescope resubmission must contain **all**
work you want graded; earlier attempts are not combined with the newest one.

**Packaging after a runtime reset:** you can run just the final cell; it downloads
the small support bundle from Hugging Face only if its helper is missing. No GPU, dataset download,
dependency installation, or training is needed for packaging. Set
`include_previous=True` and import your saved submission ZIP to recover earlier
results. If newer results exist only on Drive, mount it and restore your `RESULTS`
path first; the helper cannot recover files from a deleted runtime.

B performance uses `results/asr_submission.txt` (95 test IDs for a complete run).

The ZIP contains notebooks, available small predictions/plots, and the shared PDF.
It excludes checkpoints, pretrained weights, datasets, and audio. Keep those
separately for resuming training; do not upload them or the support bundle to
Gradescope. Locally, save your edited notebooks, `results/`, and `report.pdf`
together, then run `python make_submission.py` in that folder.


In [ ]:
include_report = True  # Upload one combined PDF, or cancel to retain an earlier one.
include_previous = True  # Merge earlier submission ZIPs; cancel if none.

# Packaging needs no training dependencies, GPU, or data download.
# Course release settings are supplied by staff; no student login is needed.
COURSE_REPO_ID = 'Louis0324/eecs183-hw4'
COURSE_REVISION = 'hw4-2026-v2'
import os
os.environ.setdefault("HW4_HF_REPO", COURSE_REPO_ID)
os.environ.setdefault("HW4_HF_REVISION", COURSE_REVISION)

def ensure_hw4_support(repo_id=None, revision='hw4-2026-v2', packager_only=False):
    """Fetch the public support bundle, or use a local/uploaded offline copy."""
    import hashlib
    from io import BytesIO
    import json
    from pathlib import Path
    import re
    from urllib.parse import quote
    from urllib.request import urlopen
    import zipfile

    required = 'make_submission.py' if packager_only else 'course_download.py'
    if Path(required).is_file():
        return
    local_bundle = Path('hw4-student.zip')
    if local_bundle.is_file():
        content = local_bundle.read_bytes()
    elif repo_id:
        if not re.fullmatch(r'[\w.-]+/[\w.-]+', repo_id):
            raise ValueError('Expected a Hugging Face repository ID like COURSE_ORG/eecs183-hw4.')
        base = f'https://huggingface.co/datasets/{repo_id}/resolve/{quote(revision, safe="")}/'
        print('Downloading the small HW4 support bundle from Hugging Face; no login is required.')
        try:
            with urlopen(base + 'release_manifest.json', timeout=30) as response:
                manifest = json.loads(response.read(1024 * 1024))
            spec = manifest['files']['hw4-student.zip']
            if not 0 < spec['bytes'] < 30 * 1024 * 1024:
                raise ValueError('Invalid support bundle size in the release manifest.')
            with urlopen(base + 'hw4-student.zip', timeout=30) as response:
                content = response.read(spec['bytes'] + 1)
            if len(content) != spec['bytes'] or hashlib.sha256(content).hexdigest() != spec['sha256']:
                raise ValueError('Support bundle checksum mismatch; rerun to download a fresh copy.')
        except Exception as error:
            raise RuntimeError('Could not download the HW4 support release. Rerun this cell; '
                               'if needed, download hw4-student.zip from the course Hugging Face repo '
                               'and upload it through the Files sidebar before retrying.') from error
    else:
        # Allows local staff testing and offline/manual use before publication.
        try:
            from google.colab import files
        except ImportError as error:
            raise FileNotFoundError('Put hw4-student.zip in this folder or configure the course Hugging Face repository.') from error
        print('Upload hw4-student.zip to load the supplied helpers.')
        uploaded = files.upload()
        archives = [value for name, value in uploaded.items() if name.lower().endswith('.zip')]
        if len(archives) != 1:
            raise ValueError('Upload exactly one hw4-student.zip support bundle, then rerun.')
        content = archives[0]

    with zipfile.ZipFile(BytesIO(content)) as archive:
        if archive.testzip() is not None:
            raise ValueError('Corrupt support ZIP; download a fresh copy.')
        for item in archive.infolist():
            path = Path(item.filename)
            if (path.is_absolute() or '..' in path.parts or
                    ((item.external_attr >> 16) & 0o170000) == 0o120000 or
                    not path.resolve().is_relative_to(Path.cwd().resolve())):
                raise ValueError('Unsafe support archive path.')
            # Browser/local notebook edits must never be replaced by starter copies.
            if item.is_dir() or path.suffix == '.ipynb' or (packager_only and item.filename != required):
                continue
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(archive.read(item))
    if not Path(required).is_file():
        raise ValueError(f'This ZIP is missing {required}; use the current hw4-student.zip support bundle.')

ensure_hw4_support(COURSE_REPO_ID, COURSE_REVISION, packager_only=True)
try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None
from make_submission import build_submission, from_colab
if colab_files is not None:
    from_colab('b', include_report=include_report, include_previous=include_previous,
               results_dir=globals().get("RESULTS", "results"))
else:
    build_submission(report="report.pdf" if include_report else None)
